# Stage 00 — Image Extractor (scan mode)

Downloads drawings for the **first `scan_limit` (10) records** from PatSeer
and saves them to `paths.raw_images/<patent_id>/fig_XX.png`.
Description text is saved to `paths.text/<patent_id>.txt`.
Downloaded thumbnails are displayed inline per patent.

In [ ]:
import sys
from pathlib import Path

# Make src/ importable from notebooks/
repo_root = Path().resolve().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

In [ ]:
from src.config_loader import load_config
from src.extractor import (
    build_driver, login_flow, iter_records,
    extract_drawings_for_record, download_images,
    make_requests_session, load_patseer_excel, save_description_text,
)

cfg = load_config()
print(f"raw_images : {cfg['paths']['raw_images']}")
print(f"text       : {cfg['paths']['text']}")
print(f"scan_limit : {cfg['extractor']['scan_limit']}")

In [ ]:
# Load Excel index (prints all column headers so you can verify names)
excel_index = load_patseer_excel(cfg['paths']['patseer_excel'])
print(f"\nIndexed {len(excel_index)} patents from Excel.")

In [ ]:
from selenium.webdriver.support.ui import WebDriverWait

scan_limit = cfg['extractor']['scan_limit']   # 10
raw_dir    = cfg['paths']['raw_images']
text_dir   = cfg['paths']['text']

driver = build_driver(cfg)
wait   = WebDriverWait(driver, 15)
login_flow(driver, cfg)   # opens browser — log in, then press Enter

In [ ]:
total_images = 0
errors = []
downloaded_patents = []

try:
    for idx in iter_records(driver, cfg, limit=scan_limit):
        patent_id, urls = extract_drawings_for_record(driver, wait, idx)

        patent_dir = raw_dir / patent_id
        if patent_dir.exists() and any(patent_dir.glob('fig_*')):
            count = len(list(patent_dir.glob('fig_*')))
            print(f"  Already downloaded {count} image(s) — skipping")
            downloaded_patents.append(patent_id)
            continue

        if not urls:
            errors.append((idx, patent_id, 'no image URLs found'))
            continue

        session = make_requests_session(driver)
        n = download_images(urls, patent_id, session, raw_dir)
        total_images += n

        if patent_id in excel_index:
            save_description_text(patent_id, excel_index[patent_id], text_dir)

        downloaded_patents.append(patent_id)

except KeyboardInterrupt:
    print('\n⏹  Interrupted.')
finally:
    driver.quit()

print(f'\nTotal images saved: {total_images}')
if errors:
    for i, pn, reason in errors:
        print(f'  Record {i}: {pn} → {reason}')

In [ ]:
# ── Display downloaded thumbnails per patent ──────────────────────────────────
from IPython.display import display
from PIL import Image
import matplotlib.pyplot as plt

THUMB_SIZE = (200, 200)

for patent_id in downloaded_patents:
    patent_dir = raw_dir / patent_id
    images = sorted(patent_dir.glob('fig_*'))
    if not images:
        continue

    print(f"\n{'─'*60}")
    print(f"  {patent_id}  ({len(images)} images)")
    print(f"{'─'*60}")

    fig, axes = plt.subplots(1, len(images), figsize=(3 * len(images), 3))
    if len(images) == 1:
        axes = [axes]

    for ax, img_path in zip(axes, images):
        img = Image.open(img_path)
        img.thumbnail(THUMB_SIZE)
        ax.imshow(img, cmap='gray' if img.mode == 'L' else None)
        ax.set_title(img_path.name, fontsize=8)
        ax.axis('off')

    plt.tight_layout()
    plt.show()